In [2]:

import torch
import sys, os, pdb
import argparse, logging
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
# print(Path(os.path.realpath(__file__)).parents /'src' / 'voxprofile' / 'src' / 'model' / 'age_sex')
target_path = Path.cwd().parent / 'src' / 'voxprofile' / 'src' / 'model' / 'age_sex'

print(target_path)
from voxprofile.src.model.age_sex.wavlm_demographics import WavLMWrapper
from voxprofile.src.model.age_sex.whisper_demographics import WhisperWrapper
# sys.path.append(os.path.join(str(Path(os.path.realpath(__file__)).parents[1])))
# sys.path.append(os.path.join(str(Path(os.path.realpath(__file__)).parents[1]), 'model', 'age_sex'))

# from wavlm_demographics import WavLMWrapper

# define logging console
import logging
logging.basicConfig(
    format='%(asctime)s %(levelname)-3s ==> %(message)s', 
    level=logging.INFO, 
    datefmt='%Y-%m-%d %H:%M:%S'
)

os.environ["MKL_NUM_THREADS"] = "1" 
os.environ["NUMEXPR_NUM_THREADS"] = "1" 
os.environ["OMP_NUM_THREADS"] = "1" 


/home/tiche/Github/Tools4Speech/src/voxprofile/src/model/age_sex


In [7]:
import os
import gc
import soundfile as sf
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from typing import List, Dict, Any, Optional
import pandas as pd

import gc
import numpy as np
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from tqdm.auto import tqdm

import torch
import sys, os, pdb
import argparse, logging
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
# print(Path(os.path.realpath(__file__)).parents /'src' / 'voxprofile' / 'src' / 'model' / 'age_sex')
target_path = Path.cwd().parent / 'src' / 'voxprofile' / 'src' / 'model' / 'age_sex'

print(target_path)
from voxprofile.src.model.age_sex.wavlm_demographics import WavLMWrapper
from voxprofile.src.model.age_sex.whisper_demographics import WhisperWrapper
@dataclass
class TransformersAgeSexModel:
    backend: str
    model: Any
    agesex_model_name: str
    device: str
    cache_dir: Optional[str]
    model_batch_size: int
    compute_type: str

SEX_UNIQUE_LABELS = ["Female", "Male"]

def load_age_sex_model(
    agesex_model_name: str = "tiantiaf/wavlm-large-age-sex",
    device: str = "auto",
    cache_dir: Optional[str] = None,
    model_batch_size: int = 16,
    backend: str = "auto",
    compute_type: Optional[str] = None,
) -> TransformersAgeSexModel:
    """Initialise and return a demographic prediction model via Voxprofile."""
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    elif device == "cuda" and not torch.cuda.is_available():
        device = "cpu"

    if compute_type is None:
        compute_type = "float16" if device == "cuda" else "float32"

    if "wavlm" in agesex_model_name:
        backend = "wavlm-large"
        model = WavLMWrapper.from_pretrained(agesex_model_name).to(device)
        model.eval() 
        
        if compute_type == "float16" and device == "cuda":
            model = model.half()

        return TransformersAgeSexModel(
            backend=backend,
            model=model,
            agesex_model_name=agesex_model_name,
            device=device,
            cache_dir=cache_dir,
            model_batch_size=model_batch_size,
            compute_type=compute_type,
        )
    
    raise ValueError(f"Unsupported model or backend: {agesex_model_name}")
def _wavlm_predict_batch_inference(
    batch_segments: List[Dict[str, Any]], 
    model_wrapper: Any
) -> List[Dict[str, Any]]:
    """Loads audio, tracks actual sample lengths, pads them dynamically, and runs batch inference."""
    device = model_wrapper.device
    model = model_wrapper.model
    
    audio_tensors = []
    lengths = []
    max_len = 0
    
    # 1. Load raw audio and track original lengths (From your working snippet)
    for seg in batch_segments:
        audio, sr = sf.read(seg["seg_filename"])
        if audio.ndim > 1:
            audio = audio[:, 0]
            
        tensor = torch.tensor(audio, dtype=torch.float32)
        audio_tensors.append(tensor)
        lengths.append(len(tensor))
        if len(tensor) > max_len:
            max_len = len(tensor)
            
    # 2. Dynamic zero-padding to make shapes uniform for stacking
    padded_tensors = []
    for tensor in audio_tensors:
        pad_size = max_len - len(tensor)
        if pad_size > 0:
            tensor = F.pad(tensor, (0, pad_size), "constant", 0.0)
        padded_tensors.append(tensor)
        
    # 3. Stack arrays and lengths into shapes expected by the model
    input_batch = torch.stack(padded_tensors).to(device)
    input_lengths = torch.tensor(lengths, dtype=torch.long).to(device)
    
    if getattr(model_wrapper, "compute_type", None) == "float16" and device == "cuda":
        input_batch = input_batch.half()
        
    # 4. Model Inference Execution with lengths
    with torch.no_grad():
        # Passing length=input_lengths directly to fix the masking IndexError!
        wavlm_age_outputs, wavlm_sex_outputs = model(input_batch, length=input_lengths)
        
        # Age extraction
        age_preds = (wavlm_age_outputs.detach().cpu().numpy() * 100).flatten()
        
        # Sex probability extraction
        sex_probs = F.softmax(wavlm_sex_outputs, dim=1)
        sex_indices = torch.argmax(sex_probs, dim=1).detach().cpu().tolist()
        
    # 5. Format structured outputs (Note: Ensure SEX_UNIQUE_LABELS is defined globally)
    batch_results = []
    for idx, sex_idx in enumerate(sex_indices):
        batch_results.append({
            "age": float(age_preds[idx]),
            "sex": SEX_UNIQUE_LABELS[sex_idx]  # Ensure this array matches your setup
        })
        
    return batch_results


def _predict_demographics_batch(
    batch: List[Dict[str, Any]],
    model: Any,
    cache: bool = False,
) -> List[Dict[str, Any]]:
    """Manages cache validation and coordinates batch routing blocks."""
    files_to_predict = []
    file_indices = []
    results = [None] * len(batch)

    for i, seg in enumerate(batch):
        txt_cache = seg["txt_cache"]
        if cache and os.path.exists(txt_cache):
            try:
                with open(txt_cache, "r", encoding="utf-8") as cache_file:
                    cached_text = cache_file.read().strip()
                if not cached_text.startswith("[AGE_SEX_PREDICTION_FAILED:"):
                    parts = cached_text.split(" | ")
                    age = float(parts[0].split(": ")[1])
                    sex = parts[1].split(": ")[1]
                    results[i] = {"age": age, "sex": sex}
                    continue
            except Exception:
                pass 
                
        files_to_predict.append(seg)
        file_indices.append(i)

    if files_to_predict:
        batch_outputs = _wavlm_predict_batch_inference(files_to_predict, model)
        
        for batch_idx, output in zip(file_indices, batch_outputs):
            results[batch_idx] = output
            
            if cache:
                cache_path = batch[batch_idx]["txt_cache"]
                with open(cache_path, "w", encoding="utf-8") as cache_file:
                    cache_file.write(f"Age: {output['age']:.1f} | Sex: {output['sex']}")

    return results


def predict_demographics_segments(
    model: Any,
    segments: pd.DataFrame,
    output_dir: str,
    cache: bool = True,
    batch_size: Optional[float] = 30.0,  # Bumped from 1.0 to 30.0 to allow actual batching
    min_duration_samples: int = 1600,
) -> Dict[str, Any]:
    """Primary pipeline executor to slice data structures and append metrics."""
    batches = batch_files(segments, output_dir, batch_size)

    predictions_map = {}
    for batch in tqdm(batches, desc=f"Processing {len(batches)} demographic batches"):
        batch_results = _predict_demographics_batch(batch, model, cache=cache)
        
        for seg, res in zip(batch, batch_results):
            predictions_map[seg["idx"]] = res

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return predictions_map

/home/tiche/Github/Tools4Speech/src/voxprofile/src/model/age_sex


In [ ]:
import os
import gc
import soundfile as sf
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from typing import List, Dict, Any, Optional
import pandas as pd

import gc
import numpy as np
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from tqdm.auto import tqdm

import torch
import sys, os, pdb
import argparse, logging
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
# print(Path(os.path.realpath(__file__)).parents /'src' / 'voxprofile' / 'src' / 'model' / 'age_sex')
target_path = Path.cwd().parent / 'src' / 'voxprofile' / 'src' / 'model' / 'age_sex'

print(target_path)
from voxprofile.src.model.age_sex.wavlm_demographics import WavLMWrapper
from voxprofile.src.model.age_sex.whisper_demographics import WhisperWrapper
@dataclass
class TransformersAgeSexModel:
    backend: str
    model: Any
    agesex_model_name: str
    device: str
    cache_dir: Optional[str]
    model_batch_size: int
    compute_type: str

SEX_UNIQUE_LABELS = ["Female", "Male"]

def load_age_sex_model(
    agesex_model_name: str = "tiantiaf/wavlm-large-age-sex",
    device: str = "auto",
    cache_dir: Optional[str] = None,
    model_batch_size: int = 16,
    backend: str = "auto",
    compute_type: Optional[str] = None,
) -> TransformersAgeSexModel:
    """Initialise and return a demographic prediction model via Voxprofile."""
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    elif device == "cuda" and not torch.cuda.is_available():
        device = "cpu"

    if compute_type is None:
        compute_type = "float16" if device == "cuda" else "float32"

    if "wavlm" in agesex_model_name:
        backend = "wavlm-large"
        model = WavLMWrapper.from_pretrained(agesex_model_name).to(device)
        model.eval() 
        
        if compute_type == "float16" and device == "cuda":
            model = model.half()

        return TransformersAgeSexModel(
            backend=backend,
            model=model,
            agesex_model_name=agesex_model_name,
            device=device,
            cache_dir=cache_dir,
            model_batch_size=model_batch_size,
            compute_type=compute_type,
        )
    
    raise ValueError(f"Unsupported model or backend: {agesex_model_name}")
    
def batch_files(
    segments: pd.DataFrame,
    output_dir: str,
    batch_size: Optional[float] = 30.0,
) ->List[List[Dict[str, Any]]]:
    os.makedirs(output_dir, exist_ok=True)
    
    segment_info = []
    for idx, row in segments.iterrows():
        seg_filename = row.get("seg_filename", f"segment_{idx}.wav")
        
        segment_info.append({
            "idx": idx,
            "speaker": row.get("speaker", "unknown"),
            "start_sec": float(row["start_sec"]),
            "end_sec": float(row["end_sec"]),
            "seg_filename": seg_filename,
        })

    max_batch_duration = (
        float(batch_size) if batch_size and batch_size > 0 else float("inf")
    )

    batches: List[List[Dict[str, Any]]] = []
    current_batch: List[Dict[str, Any]] = []
    current_duration = 0.0

    for seg in segment_info:
        seg_duration = float(seg["end_sec"] - seg["start_sec"])

        if seg_duration > max_batch_duration:
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_duration = 0.0
            batches.append([seg])
            continue

        if current_batch and current_duration + seg_duration > max_batch_duration:
            batches.append(current_batch)
            current_batch = [seg]
            current_duration = seg_duration
        else:
            current_batch.append(seg)
            current_duration += seg_duration

    if current_batch:
        batches.append(current_batch)

    return batches

/home/tiche/Github/Tools4Speech/src/voxprofile/src/model/age_sex


In [23]:
def _predict_age_sex_batch(
    batch: List[Dict[str, Any]],
    model: TransformersAgeSexModel,
    cache: bool = False,
) -> List[Dict[str, Any]]:
    """Manages cache validation and coordinates batch routing blocks."""
    files_to_predict = []
    file_indices = []
    results = [None] * len(batch)
    for i, seg in enumerate(batch):
        cache = seg['seg_filename'].replace(".wav", "_demographics.txt")
        print(cache)
        if cache and os.path.exists(txt_cache):
            try:
                with open(txt_cache, "r", encoding="utf-8") as cache_file:
                    cached_text = cache_file.read().strip()
                if not cached_text.startswith("[AGE_SEX_PREDICTION_FAILED:"):
                    # Cache format pattern: "Age: 32.5 | Sex: Male"
                    parts = cached_text.split(" | ")
                    age = float(parts[0].split(": ")[1])
                    sex = parts[1].split(": ")[1]
                    results[i] = {"age": age, "sex": sex}
                    continue
            except Exception:
                pass # Stale cache configuration fallback
                
        files_to_predict.append(seg)
        file_indices.append(i)

    # Execute processing if active inputs exist
    if files_to_predict:
        batch_outputs = _wavlm_predict_batch_inference(files_to_predict, model)
        
        for batch_idx, output in zip(file_indices, batch_outputs):
            results[batch_idx] = output
            
            if cache:
                cache_path = batch[batch_idx]["txt_cache"]
                with open(cache_path, "w", encoding="utf-8") as cache_file:
                    cache_file.write(f"Age: {output['age']:.1f} | Sex: {output['sex']}")

    return results

In [9]:
import io
import pandas as pd
from typing import Dict

raw_data = """speaker	start_sec	end_sec	seg_filename	duration_sec	transcription	type
P1	3.02	39.6	../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20.wav	36.58	Jeg kan faktisk godt lide at lave mad...	turn
P1	47.76	48.23	../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p20.wav	0.47	Ja.	backchannel
P1	56.22	56.65	../outputs/dyad/P1/P1_seg_2_56.22_56.65_pad0p20.wav	0.43	Ja.	backchannel
P2	0.78	2.6	../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav	1.82	Kan du lide at lave mad?	turn
P2	41.01	59.98	../outputs/dyad/P2/P2_seg_1_41.01_59.98_pad0p20.wav	18.97	Det genkender jeg faktisk vældig godt...	turn"""

df = pd.read_csv(io.StringIO(raw_data), sep="\t")
df["duration_sec"] = df["duration_sec"].round(2)

data = df.to_dict(orient="records") 

segments_by_speaker: Dict[str, pd.DataFrame] = {
    str(speaker): group_df.reset_index(drop=True) 
    for speaker, group_df in df.groupby("speaker")
}
segments_by_speaker

{'P1':   speaker  start_sec  end_sec  \
 0      P1       3.02    39.60   
 1      P1      47.76    48.23   
 2      P1      56.22    56.65   
 
                                         seg_filename  duration_sec  \
 0  ../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20...         36.58   
 1  ../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p2...          0.47   
 2  ../outputs/dyad/P1/P1_seg_2_56.22_56.65_pad0p2...          0.43   
 
                               transcription         type  
 0  Jeg kan faktisk godt lide at lave mad...         turn  
 1                                       Ja.  backchannel  
 2                                       Ja.  backchannel  ,
 'P2':   speaker  start_sec  end_sec  \
 0      P2       0.78     2.60   
 1      P2      41.01    59.98   
 
                                         seg_filename  duration_sec  \
 0  ../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav          1.82   
 1  ../outputs/dyad/P2/P2_seg_1_41.01_59.98_pad0p2...         18.97   
 
    

In [18]:
agesex_model_batch_size = 10
agesex_model_name = "tiantiaf/wavlm-large-age-sex"
device = "auto"
model = load_age_sex_model(
    agesex_model_name=agesex_model_name,
    device=device,
    model_batch_size=agesex_model_batch_size,
)
final_results = []

In [ ]:
for speaker, segments in  segments_by_speaker.items():
    print(f"Processing demographics for speaker: {speaker} with {len(segments)} segments")
    predictions_map = predict_demographics_segments(model, segments, output_dir="outputs/demographics", cache=False)
    
    for idx, row in segments.iterrows():
        # Look up the prediction using the true pandas DataFrame row index (idx)
        pred = predictions_map.get(idx, {"age": np.nan, "sex": "Unknown"})
        
        final_results.append({
            "speaker": row["speaker"],
            "start_sec": row["start_sec"],
            "end_sec": row["end_sec"],
            "estimated_age": pred["age"],
            "estimated_sex": pred["sex"],
            "file_source": row.get("seg_filename", f"segment_{idx}.wav")
        })


Processing demographics for speaker: P1 with 3 segments


Processing 2 demographic batches:   0%|          | 0/2 [00:00<?, ?it/s]

/home/tiche/.conda/envs/vdt/lib/python3.10/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Processing demographics for speaker: P2 with 2 segments


Processing 1 demographic batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
final_results

[{'speaker': 'P1',
  'start_sec': 3.02,
  'end_sec': 39.6,
  'estimated_age': 27.48085594177246,
  'estimated_sex': 'Male',
  'file_source': '../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20.wav'},
 {'speaker': 'P1',
  'start_sec': 47.76,
  'end_sec': 48.23,
  'estimated_age': 26.68528175354004,
  'estimated_sex': 'Male',
  'file_source': '../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p20.wav'},
 {'speaker': 'P1',
  'start_sec': 56.22,
  'end_sec': 56.65,
  'estimated_age': 29.71879768371582,
  'estimated_sex': 'Male',
  'file_source': '../outputs/dyad/P1/P1_seg_2_56.22_56.65_pad0p20.wav'},
 {'speaker': 'P2',
  'start_sec': 0.78,
  'end_sec': 2.6,
  'estimated_age': 27.58344078063965,
  'estimated_sex': 'Male',
  'file_source': '../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav'},
 {'speaker': 'P2',
  'start_sec': 41.01,
  'end_sec': 59.98,
  'estimated_age': 23.337228775024414,
  'estimated_sex': 'Male',
  'file_source': '../outputs/dyad/P2/P2_seg_1_41.01_59.98_pad0p20.wav'}]

In [8]:
def predict_demographics_segments(
    model: TransformersAgeSexModel,
    segments: pd.DataFrame,
    output_dir: str,
    cache: bool = True,
    batch_size: Optional[float] = 1.0,
    min_duration_samples: int = 1600,
) -> List[Dict[str, Any]]:
    """Primary pipeline executor to slice data structures and append metrics."""
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Re-map rows to safe workspace dictionaries
    segment_info = []
    for idx, row in segments.iterrows():
        
        # Fallback tracking if pre-sliced files don't exist yet
        seg_filename = row.get("seg_filename", f"segment_{idx}.wav")
        print(idx, seg_filename)
        txt_cache = os.path.join(output_dir, os.path.basename(seg_filename).replace(".wav", "_demographics.txt"))
        
        segment_info.append({
            "idx": idx,
            "speaker": row.get("speaker", "unknown"),
            "start_sec": float(row["start_sec"]),
            "end_sec": float(row["end_sec"]),
            "seg_filename": seg_filename,
            "txt_cache": txt_cache,
        })

    # 2. Build time-bucketed accumulation windows
    max_batch_duration = (
        float(batch_size) if batch_size and batch_size > 0 else float("inf")
    )

    batches: List[List[Dict[str, Any]]] = []
    current_batch: List[Dict[str, Any]] = []
    current_duration = 0.0

    for seg in segment_info:
        seg_duration = float(seg["end_sec"] - seg["start_sec"])

        if seg_duration > max_batch_duration:
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_duration = 0.0
            batches.append([seg])
            continue

        if current_batch and current_duration + seg_duration > max_batch_duration:
            batches.append(current_batch)
            current_batch = [seg]
            current_duration = seg_duration
        else:
            current_batch.append(seg)
            current_duration += seg_duration

    if current_batch:
        batches.append(current_batch)

    # 3. Process each batch window sequentially
    predictions_map = {}
    for batch in tqdm(batches, desc=f"Processing {len(batches)} demographic batches"):
        batch_results = _predict_age_sex_batch(batch, model, cache=cache)
        
        for seg, res in zip(batch, batch_results):
            predictions_map[seg["idx"]] = res

        # Force garbage collection loops to prevent memory leakage
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # 4. Compile predictions back into original input alignment
    final_results = []
    for seg in segment_info:
        pred = predictions_map.get(seg["idx"], {"age": np.nan, "sex": "Unknown"})
        final_results.append({
            "speaker": seg["speaker"],
            "start_sec": seg["start_sec"],
            "end_sec": seg["end_sec"],
            "estimated_age": pred["age"],
            "estimated_sex": pred["sex"],
            "file_source": seg["seg_filename"]
        })

    return final_results

In [7]:
wavlm_model = WavLMWrapper.from_pretrained("tiantiaf/wavlm-large-age-sex").to("cpu")
wavlm_model.eval()
audio, sr = sf.read("../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav")
data = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to("cpu")
sex_unique_labels = ["Female", "Male"]
wavlm_age_outputs, wavlm_sex_outputs = wavlm_model(data)

age_pred = wavlm_age_outputs.detach().cpu().numpy() * 100

sex_prob = F.softmax(wavlm_sex_outputs, dim=1)
print(sex_unique_labels[torch.argmax(sex_prob).detach().cpu().item()])
print(age_pred)




Male
[[28.938183]]


In [12]:
from voxprofile.src.model.emotion.wavlm_emotion import WavLMWrapper
emotion_list = [
    'Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 
    'Neutral', 'Sadness', 'Surprise', 'Other'
]
model = WavLMWrapper.from_pretrained("tiantiaf/wavlm-large-categorical-emotion").to('cpu')
model.eval()
logits = model(data, return_feature=True)    
print(logits[0])
# Probability and output
emotion_prob = F.softmax(logits[0], dim=1)
print(emotion_list[torch.argmax(emotion_prob).detach().cpu().item()])

0
tensor([[-14.2155, -11.3148, -13.7367, -13.7327, -10.6351,  -8.3523,  -9.9015,
         -11.0965,  -9.0453]], grad_fn=<AddmmBackward0>)
Neutral


In [15]:
import soundfile as sf
import torch
import torch.nn.functional as F

# 1. Initialize model
wavlm_model = WavLMWrapper.from_pretrained("tiantiaf/wavlm-large-categorical-emotion").to("cpu")
wavlm_model.eval()

# 2. Simulate loading multiple audio files of different lengths
file_paths = [
    "../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p20.wav",
    "../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20.wav"
]

audio_tensors = []
lengths = []
max_len = 0

for path in file_paths:
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio[:, 0]
    
    tensor = torch.tensor(audio, dtype=torch.float32)
    audio_tensors.append(tensor)
    lengths.append(len(tensor))
    if len(tensor) > max_len:
        max_len = len(tensor)

# 3. Dynamic zero-padding to make them uniform for stacking
padded_tensors = []
for tensor in audio_tensors:
    pad_size = max_len - len(tensor)
    if pad_size > 0:
        tensor = F.pad(tensor, (0, pad_size), "constant", 0.0)
    padded_tensors.append(tensor)

# 4. Stack arrays into shapes expected by the model
input_batch = torch.stack(padded_tensors).to("cpu")  # Shape: [Batch_Size, Max_Samples]
input_lengths = torch.tensor(lengths, dtype=torch.long).to("cpu")  # Shape: [Batch_Size]

# 5. Run Batch Inference using the correct 'length' argument
with torch.no_grad():
    # Pass the batched tensor as x, and the raw sample lengths as length
    logits = model( input_batch, length=input_lengths, return_feature=True)
emotion_prob = F.softmax(logits[0], dim=1)
emo_indices = torch.argmax(emotion_prob, dim=1).detach().cpu().tolist()

print("\n--- Emotion Results ---")
for idx, emo_idx in enumerate(emo_indices):
    print(f"File {idx} -> Emotion: {emotion_list[emo_idx]}")

0
1

--- Emotion Results ---
File 0 -> Emotion: Neutral
File 1 -> Emotion: Neutral


In [91]:
import os
import soundfile as sf
import torch
import torch.nn.functional as F
from typing import List, Dict, Any

# Label List from your documentation
emotion_label_list = [
    'Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 
    'Neutral', 'Sadness', 'Surprise', 'Other'
]

def predict_emotion_batch(
    file_paths: List[str], 
    emo_model: torch.nn.Module, 
    device: str = "cpu"
) -> List[Dict[str, Any]]:
    """Loads a batch of audio files, pads them to match the longest file 

    (up to 15s max), and runs batch inference.
    """
    # 16kHz * 15 seconds max cap from documentation
    MAX_SAMPLES = 15 * 16000 
    
    audio_tensors = []
    lengths = []
    max_len = 0

    # 1. Load and prepare each audio file
    for path in file_paths:
        audio, sr = sf.read(path)
        if audio.ndim > 1:
            audio = audio[:, 0]  # Ensure mono channel
            
        # Optional: Resample here if your files aren't 16000Hz
        
        # Enforce max duration limit (15 seconds) by slicing trailing audio
        tensor = torch.tensor(audio, dtype=torch.float32)[:MAX_SAMPLES]
        
        audio_tensors.append(tensor)
        lengths.append(len(tensor))
        if len(tensor) > max_len:
            max_len = len(tensor)

    # 2. Dynamic zero-padding to make the batch matrix uniform
    padded_tensors = []
    for tensor in audio_tensors:
        pad_size = max_len - len(tensor)
        if pad_size > 0:
            tensor = F.pad(tensor, (0, pad_size), "constant", 0.0)
        print(pad_size)
        padded_tensors.append(tensor)

    # 3. Stack arrays and send to the target execution device
    input_batch = torch.stack(padded_tensors).to(device)
    input_lengths = torch.tensor(lengths, dtype=torch.long).to(device)
    print(input_lengths)
    # 4. Model Inference Execution using the 6-value tuple unpack
    with torch.no_grad():
        logits = emo_model(
            input_batch, 
            return_feature=True, 
            length=input_lengths
        )

    # 5. Calculate probabilities across rows (dim=1)
    emotion_probs = F.softmax(logits[0], dim=1)
    predicted_indices = torch.argmax(emotion_probs, dim=1).detach().cpu().tolist()

    # 6. Format structured results
    batch_results = []
    for idx, emo_idx in enumerate(predicted_indices):
        batch_results.append({
            "file_path": file_paths[idx],
            "predicted_emotion": emotion_label_list[emo_idx],
            "probabilities": emotion_probs[idx].detach().cpu().numpy().tolist(),
            # "embedding": embedding[idx].detach().cpu().numpy()  # Isolated segment embedding vectors
        })

    return batch_results

In [95]:
# 1. Initialize and secure eval mode
device = "cpu"  # or "cuda"
emo_model = WavLMWrapper.from_pretrained("tiantiaf/wavlm-large-categorical-emotion").to(device)
emo_model.eval()

# 2. Define files
files_to_process = [
    "../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p20.wav",
    "../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav"
]

# 3. Predict
results = predict_emotion_batch(files_to_process, emo_model, device=device)

# 4. Inspect outputs
for res in results:
    print(f"File: {os.path.basename(res['file_path'])} -> Emotion: {res['predicted_emotion']}")

64800
0
tensor([ 41760, 106560])
File: P1_seg_1_47.76_48.23_pad0p20.wav -> Emotion: Neutral
File: P2_seg_0_0.78_2.60_pad0p20.wav -> Emotion: Neutral


In [4]:

def predict_emotion_segments(
    model: Any,
    segments: pd.DataFrame,
    output_dir: str,
    cache: bool = True,
    batch_size: Optional[float] = 30.0,  # Bumped from 1.0 to 30.0 to allow actual batching
    min_duration_samples: int = 1600,
) -> List[Dict[str, Any]]:
    """Primary pipeline executor to slice data structures and append metrics."""
    
    batches = batch_files(segments, output_dir, batch_size)
    predictions_map = {}
    for batch in tqdm(batches, desc=f"Processing {len(batches)} demographic batches"):
        batch_results = _predict_demographics_batch(batch, model, cache=cache)
        
        for seg, res in zip(batch, batch_results):
            predictions_map[seg["idx"]] = res

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    final_results = []
    for seg in segment_info:
        pred = predictions_map.get(seg["idx"], {"age": np.nan, "sex": "Unknown"})
        final_results.append({
            "speaker": seg["speaker"],
            "start_sec": seg["start_sec"],
            "end_sec": seg["end_sec"],
            "estimated_age": pred["age"],
            "estimated_sex": pred["sex"],
            "file_source": seg["seg_filename"]
        })

    return final_results